# Crossed rigid fibers

This is the smallest complete Python-to-GPU TANGLE workflow. Two straight fibers begin at a 90-degree crossing, are inserted into the cell, translated apart without changing shape, and exported for OVITO and a downstream BPM solver.

In [ ]:
from pathlib import Path
import tangle

output = Path("output")
output.mkdir(exist_ok=True)

## Build a detached collection

A `FiberCollection` is local geometry that has not yet been inserted into a simulation cell. Because no separate `rest_centerline` is supplied, each straight centerline is also its stress-free shape.

In [ ]:
fiber = tangle.Material("fiber", diameter=0.05)
crossing = tangle.FiberCollection("orthogonal crossing")
crossing.add_fiber(
    [[-0.35, 0.0, 0.0], [0.35, 0.0, 0.0]],
    fiber,
    tags={"family": "x"},
)
crossing.add_fiber(
    [[0.0, -0.35, 0.0], [0.0, 0.35, 0.0]],
    fiber,
    tags={"family": "y"},
)
crossing

## Insert, then request relaxation

Insertion applies a rigid translation to both the placed and rest descriptions. `recipe.relax()` adds an operation; it does not run the solver yet.

In [ ]:
recipe = tangle.Recipe(tangle.Cell([1.0, 1.0, 1.0]))
inserted = recipe.insert(crossing, translation=[0.5, 0.5, 0.5])
recipe.relax(maximum_iterations=2_000)
recipe.operations()

## Choose solver settings

Rigid translation gives every fiber one translation vector, preserving its exact centerline. All settings start from Rust defaults and remain editable from Python.

In [ ]:
settings = tangle.RelaxationSettings()
settings.motion_model = "rigid_translation"
settings.penetration_tolerance = 1.0e-6
settings.max_iterations = 4_000
settings.max_step = 0.01
settings.to_dict()

In [ ]:
result = recipe.run(settings)
print(inserted)
print(result)
result.centerlines()

## Export the converged assembly

The OVITO recipe configures oriented spherocylinders. The BPM data file contains exact-segment spherocylinders and fiber-continuity bonds.

In [ ]:
result.write_ovito(
    output / "crossed_fibers.dump",
    view_script_path=output / "crossed_fibers_view.py",
    session_path=output / "crossed_fibers.ovito",
)
capsules, bonds = result.export_bpm(
    output / "crossed_fibers_capsules.data",
    mode="spherocylinders-exact",
    density=1_800.0,
)
print(f"exported {capsules} capsules and {bonds} bonds")